# Exemplar: a task, SGLD training, and the mean-field solve, watched live

A linear teacher ($x = Z W_X^\top$, $y = Z W_Y^\top$ for $k$ latents $Z$) is learned by a small
network trained with SGLD at inverse temperature $\beta$, so the trained net is a sample from
the distribution $e^{-\beta \mathcal L}$. The mean-field theory of that distribution is a saddle
point in the layer kernels $\Phi_l$ and their conjugates $\hat\Phi_l$, found by an outer
descent on $\Phi$ and an inner ascent on $\hat\Phi$, with the single-site expectations
estimated by a sampler. The solve is shown live: the kernels, how well the iteration is
converging, and whether the sampler can be trusted.

Set `CACHE_ROOT` in the first code cell. Training and the solve are cached under it in
directories named by hashes of the settings: re-running reloads a finished step, resumes an
interrupted one, and changing a solver setting re-solves without retraining.

**Why the default sizes are tiny.** Everything runs on a CPU in about a minute per sampler:
$P = 24$ points, 6 features, 2 latents and one hidden layer of 128 units. The prior-proposal
importance sampler (`SAMPLER = "importance"`) is a small-$P$, mild-tilt tool: its effective
sample size collapses as $P$ or $\beta$ grow, and the solver then raises
`InadequateSamplerError` rather than continue on a degenerate estimate. At $P = 24$ and
$\beta = 10$ it works; with two hidden layers, at $P = 100$ or at $\beta = 10^3$ it did not in the trial runs,
and HMC (`SAMPLER = "hmc"`) is the sampler to use there. The net is therefore a hot one
($\beta = 10$), so its train MSE sits well above the theory's, which describes the mean
predictor of the distribution; the quantities to compare are the kernels.

## 1. Task and network

Every setting of the task and the network, and the loss weights, in one cell.

In [ ]:
CACHE_ROOT = None  # directory for the cached runs; None reads $FCT_CACHE_ROOT
# task: a linear teacher with m features and k latents
P = 24  # samples; all of them enter the solve
M_IN = 6  # m = D_in: features of the input x and of the output y
K_LATENTS = 2  # k latents, fewer than m
DATA_SEED = 0
# network
N = 128  # hidden units per layer
L = 1  # hidden layers
PHI = "tanh"  # "tanh", "relu" or "sigmoid"
G = 1.0  # hidden-to-hidden gain
NET_SEED = 0
# loss weights
WEIGHT_LAMBDA = 1e-2  # weight penalty
RATE_LAMBDA = 1e-3  # rate penalty

## 2. Training

SGLD takes a gradient step plus Gaussian noise; `BETA` sets the noise. The net is trained through the `fct` cache, so a re-run is instant.

In [ ]:
OPTIMIZER = "sgld"  # "sgld" (Langevin noise of variance 2 * LR / BETA per step) or "adam"
BETA = 10.0  # inverse temperature: read by SGLD, and the beta of the mean-field theory
LR = 3e-2
N_EPOCHS = 10_000
TRAIN_SEED = 0
TRAIN_LOG_INT = 500  # epochs between logged losses

In [ ]:
import matplotlib as mpl

from fct.draw import fig_example_outputs, fig_loss_curves
from fct.exemplar import (
    LiveView,
    cache_root_from_env,
    empirical_kernel_figure,
    example_outputs_of,
    exemplar_config,
    kernel_comparison,
    solve_exemplar,
    solve_summary,
    train_exemplar,
    with_solver,
)
from fct.report import show

mpl.rcParams["figure.dpi"] = 100
RUNS_ROOT = CACHE_ROOT or cache_root_from_env()

In [ ]:
CFG = exemplar_config(
    P=P,
    m=M_IN,
    k=K_LATENTS,
    N=N,
    L=L,
    phi=PHI,
    g=G,
    data_seed=DATA_SEED,
    net_seed=NET_SEED,
    weight_lambda=WEIGHT_LAMBDA,
    rate_lambda=RATE_LAMBDA,
    optimizer=OPTIMIZER,
    beta=BETA,
    lr=LR,
    n_epochs=N_EPOCHS,
    train_seed=TRAIN_SEED,
    log_int=TRAIN_LOG_INT,
    save_int=TRAIN_LOG_INT,
)
DATASET, NET, HISTORY, CACHE_DIR = train_exemplar(CFG, RUNS_ROOT)
show([f"Trained net cached in `{CACHE_DIR.name}`."])

The training loss, the net's outputs on a few training points, and its empirical kernels $\Phi_l = \langle \psi\psi^\top \rangle$ at the end of training.

In [ ]:
show(
    [
        fig_loss_curves(HISTORY, title=f"{OPTIMIZER}, beta = {BETA:g}"),
        fig_example_outputs("linear_teacher", example_outputs_of(NET, DATASET, P), CFG.data),
        empirical_kernel_figure(NET, DATASET, CFG),
    ]
)

## 3. Mean-field solve

The solver settings; the problem itself (task, net, $\beta$) is fixed above.

In [ ]:
SAMPLER = "hmc"  # "hmc" (persistent chains) or "importance" (draws from the prior)
N_OUTER = 40  # outer descent steps on the kernels
PLOT_EVERY = 5  # refresh the live figures every this many outer steps
SOLVER_SEED = 0
SOLVE_LOG_INT = 1  # outer steps between log lines
# HMC only
N_CHAINS = 512
WARMUP_TRANS = 50  # transitions that let the chains equilibrate before the first estimate
# importance sampling only
N_IMPORTANCE_SAMPLES = 4096  # draws per transition

The live figures, refreshed every `PLOT_EVERY` outer steps: the kernels $\Phi_l$ and $\hat\Phi_l$; solver health (residuals, step sizes, backtracking, train MSE of the theory against the net); and sampler health (HMC acceptance, step size, divergences and ESS, or the importance ESS and the largest weight share).

In [ ]:
SOLVE_CFG = with_solver(
    CFG,
    sampler=SAMPLER,
    n_outer=N_OUTER,
    seed=SOLVER_SEED,
    log_int=SOLVE_LOG_INT,
    n_chains=N_CHAINS,
    warmup_trans=WARMUP_TRANS,
    n_importance_samples=N_IMPORTANCE_SAMPLES,
    sample_dtype="float64",
)
VIEW = LiveView(tail=6)  # replaces the figures at every refresh, keeps the last log lines
RESULT = solve_exemplar(SOLVE_CFG, DATASET, NET, RUNS_ROOT, PLOT_EVERY, VIEW.show, log=VIEW.log)
show([solve_summary(RESULT.history, SAMPLER)])

## The solved kernels next to the trained net's

In [ ]:
show([kernel_comparison(RESULT, NET, DATASET, SOLVE_CFG)])